In [25]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic(base_url="https://www.xiaoyaoapi.com")
model = "claude-sonnet-5"

In [26]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None):
    params = {"model": model, "max_tokens": 1000, "messages": messages}

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return "".join([block.text for block in message.content if block.type == "text"])

In [27]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```
请直接输出 JSON 数组本身，不要用 ```json 代码块包裹，不要任何多余说明文字。
* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)
    return json.loads(text)

In [29]:
dataset = generate_dataset()
print(dataset)

[{'task': 'Write a Python function that takes an AWS S3 bucket name and validates whether it meets S3 bucket naming rules (3-63 characters, lowercase letters, numbers, hyphens, and periods, must start and end with a letter or number).'}, {'task': 'Write a regular expression that matches and extracts the AWS account ID, region, and resource ID from an ARN string, e.g. arn:aws:ec2:us-east-1:123456789012:instance/i-0abcd1234efgh5678.'}, {'task': "Write a JSON object representing an IAM policy that grants read-only access (GetObject and ListBucket) to a specific S3 bucket named 'my-example-bucket'."}]


In [30]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)